<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/03_extraccion_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Extracción de embeddings con encoders congelados

Se calculan **una sola vez** los vectores de cada película y se guardan en Drive. Todo lo que
sigue (cabezas, fusión, diagnóstico) trabaja sobre estos archivos, nunca sobre pósters o texto crudo.

| Archivo | Encoder | Entrada | Dim | Uso |
|---|---|---|---|---|
| `clip_img.npy` | CLIP ViT-B/32 (visión + proyección) | póster rellenado a cuadrado | 512 | clasificación **y** brecha modal |
| `clip_txt.npy` | CLIP ViT-B/32 (texto + proyección) | sinopsis en fragmentos de 77 tokens, promediados | 512 | brecha modal (mismo espacio que `clip_img`) |
| `distilbert_txt.npy` | DistilBERT base uncased | sinopsis completa (hasta 512 tokens), mean pooling | 768 | clasificación |

La fila `i` de cada archivo corresponde a la fila `i` de `mmimdb_meta.parquet` (= `fila_hdf5`).

**Requiere GPU** (Entorno de ejecución → T4). Tiempo aproximado: ~15 min, la mayor parte leyendo el HDF5.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q kagglehub
import os, json, random, datetime
import h5py, numpy as np, pandas as pd, torch, transformers
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device, "| transformers", transformers.__version__, "| torch", torch.__version__)

DATA = '/content/drive/MyDrive/proyecto_multimodal/data'
EMB = '/content/drive/MyDrive/proyecto_multimodal/embeddings'
os.makedirs(EMB, exist_ok=True)

df = pd.read_parquet(f'{DATA}/mmimdb_meta.parquet')
assert (df['fila_hdf5'].values == np.arange(len(df))).all(), "el parquet debe estar en orden del HDF5"
N = len(df)
print(df.shape)

## 1. Pósters: del HDF5 a la entrada de CLIP

Los pósters están en formato VGG (BGR, media restada, 256×160). Se convierten a RGB con
`poster_a_rgb` (igual que en el notebook 01) y se **rellenan a 256×256** antes de que CLIP los
redimensione a 224×224. Así no se recorta nada: el *center crop* por defecto de CLIP cortaría
48 px arriba y abajo, justo donde suelen estar el título y las caras.

El relleno usa el **color medio de CLIP**: después de normalizar queda en ≈ 0, o sea, la región
rellenada no aporta señal.

In [ ]:
import kagglehub
ruta = kagglehub.dataset_download("johnarevalo/mmimdb")
f = h5py.File(os.path.join(ruta, 'multimodal_imdb.hdf5'), 'r')
assert f['images'].shape[0] == N

def poster_a_rgb(img):
    # (3, 256, 160) en formato VGG (BGR, media restada) -> (256, 160, 3) RGB uint8
    medias_bgr = np.array([103.939, 116.779, 123.68], dtype=np.float32)[:, None, None]
    rgb = (img.astype(np.float32) + medias_bgr)[::-1]
    return np.clip(np.rint(np.transpose(rgb, (1, 2, 0))), 0, 255).astype(np.uint8)

MEDIA_CLIP = np.array([0.48145466, 0.4578275, 0.40821073])   # media RGB de entrenamiento de CLIP
COLOR_RELLENO = np.rint(MEDIA_CLIP * 255).astype(np.uint8)

def rellenar_cuadrado(rgb):
    # (H, W, 3) -> (L, L, 3) con L = max(H, W), imagen centrada
    h, w, _ = rgb.shape
    lado = max(h, w)
    out = np.empty((lado, lado, 3), dtype=np.uint8)
    out[:] = COLOR_RELLENO
    y0, x0 = (lado - h) // 2, (lado - w) // 2
    out[y0:y0 + h, x0:x0 + w] = rgb
    return out

# Revisión visual: original vs rellenado
fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for k in range(4):
    rgb = poster_a_rgb(f['images'][k])
    axes[0, k].imshow(rgb); axes[0, k].set_title(f"original {rgb.shape[:2]}", fontsize=9)
    sq = rellenar_cuadrado(rgb)
    axes[1, k].imshow(sq); axes[1, k].set_title(f"rellenado {sq.shape[:2]}", fontsize=9)
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()

## 2. CLIP imagen → `clip_img.npy`

Se usa la salida **proyectada** (512-d): es el espacio compartido donde CLIP compara imagen y
texto, necesario para la brecha modal. Se calcula a mano (`vision_model` → `visual_projection`)
para no depender de cómo cada versión de `transformers` implementa `get_image_features`.

Se guarda **sin normalizar**: la norma L2 se aplica después, cuando haga falta.

In [ ]:
from transformers import CLIPModel, CLIPImageProcessor, CLIPTokenizer

CLIP_ID = 'openai/clip-vit-base-patch32'
clip = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()
proc_img = CLIPImageProcessor.from_pretrained(CLIP_ID)
tok_clip = CLIPTokenizer.from_pretrained(CLIP_ID)

@torch.no_grad()
def clip_imagen(imgs_rgb):
    # lista de (L, L, 3) uint8 -> (B, 512) float32
    pix = proc_img(images=imgs_rgb, return_tensors='pt')['pixel_values'].to(device)
    pooled = clip.vision_model(pixel_values=pix).pooler_output
    return clip.visual_projection(pooled).float().cpu().numpy()

# Chequeo: el procesador no debe recortar nada en una imagen cuadrada
px = proc_img(images=[rellenar_cuadrado(poster_a_rgb(f['images'][0]))], return_tensors='pt')['pixel_values']
print("Entrada a CLIP:", tuple(px.shape))

In [ ]:
BLOQUE = 512   # filas del HDF5 por lectura (lectura contigua = rápida)
ruta_img = f'{EMB}/clip_img.npy'

if os.path.exists(ruta_img):
    clip_img = np.load(ruta_img)
    print("Ya existía, se carga:", clip_img.shape)
else:
    clip_img = np.zeros((N, 512), dtype=np.float32)
    for a in range(0, N, BLOQUE):
        b = min(a + BLOQUE, N)
        crudo = f['images'][a:b]
        imgs = [rellenar_cuadrado(poster_a_rgb(x)) for x in crudo]
        for s in range(0, len(imgs), 128):
            clip_img[a + s:a + s + 128] = clip_imagen(imgs[s:s + 128])
        if (a // BLOQUE) % 10 == 0:
            print(f"{b}/{N}")
    np.save(ruta_img, clip_img)
    print("Guardado:", ruta_img, clip_img.shape)

## 3. CLIP texto → `clip_txt.npy`

CLIP solo ve **77 tokens** y el 70% de las sinopsis los supera (notebook 02). En vez de truncar,
cada sinopsis se corta en fragmentos de 75 tokens (+ inicio y fin = 77), cada fragmento se
codifica y se normaliza, y se **promedian** los fragmentos de la misma película.

Se promedian vectores ya normalizados para que todos los fragmentos pesen igual; el resultado
queda con norma ≤ 1 (más chica cuanto más distintos sean los fragmentos).

In [ ]:
UTILES = tok_clip.model_max_length - 2   # 75 tokens de contenido por fragmento
BOS, EOS, PAD = tok_clip.bos_token_id, tok_clip.eos_token_id, tok_clip.pad_token_id

def fragmentar_clip(textos):
    # Devuelve ids (F, 77), máscara (F, 77) y dueño (F,): a qué película pertenece cada fragmento
    # El aviso "sequence length is longer than ... (N > 77)" es esperado: se fragmenta abajo
    ids_txt = tok_clip(textos, add_special_tokens=False)['input_ids']
    ids, masc, duenio = [], [], []
    for i, t in enumerate(ids_txt):
        t = t or [EOS]   # sinopsis vacía: un fragmento mínimo
        for s in range(0, len(t), UTILES):
            frag = [BOS] + t[s:s + UTILES] + [EOS]
            relleno = UTILES + 2 - len(frag)
            ids.append(frag + [PAD] * relleno)
            masc.append([1] * len(frag) + [0] * relleno)
            duenio.append(i)
    return np.array(ids), np.array(masc), np.array(duenio)

@torch.no_grad()
def clip_texto(ids, masc):
    ids = torch.as_tensor(ids, device=device); masc = torch.as_tensor(masc, device=device)
    pooled = clip.text_model(input_ids=ids, attention_mask=masc).pooler_output
    return clip.text_projection(pooled).float().cpu().numpy()

def promediar_por_duenio(vecs, duenio, n):
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    suma = np.zeros((n, vecs.shape[1]), dtype=np.float64)
    np.add.at(suma, duenio, vecs)
    return (suma / np.bincount(duenio, minlength=n)[:, None]).astype(np.float32)

# Chequeo: un fragmento de un texto corto debe dar lo mismo que el tokenizador estándar
ids, masc, _ = fragmentar_clip(["a detective returns home"])
ref = tok_clip(["a detective returns home"], padding='max_length', max_length=77)
assert ids[0].tolist() == ref['input_ids'][0], "la fragmentación no reproduce el tokenizador"
print("Fragmentación OK")

In [ ]:
ruta_txt = f'{EMB}/clip_txt.npy'

if os.path.exists(ruta_txt):
    clip_txt = np.load(ruta_txt)
    print("Ya existía, se carga:", clip_txt.shape)
else:
    ids, masc, duenio = fragmentar_clip(df['sinopsis'].tolist())
    print(f"{len(ids)} fragmentos para {N} películas ({len(ids) / N:.2f} por película)")
    vecs = np.zeros((len(ids), 512), dtype=np.float32)
    for s in range(0, len(ids), 512):
        vecs[s:s + 512] = clip_texto(ids[s:s + 512], masc[s:s + 512])
    clip_txt = promediar_por_duenio(vecs, duenio, N)
    np.save(ruta_txt, clip_txt)
    print("Guardado:", ruta_txt, clip_txt.shape)

## 4. DistilBERT → `distilbert_txt.npy`

DistilBERT acepta hasta 512 tokens: solo el 0.5% de las sinopsis se trunca. Se usa **mean
pooling** sobre los tokens reales (sin padding) de la última capa. Se descarta el token `[CLS]`
porque DistilBERT no se entrenó con una tarea que le dé significado a ese token (no tiene NSP),
y el promedio suele representar mejor la frase en modelos congelados.

Se ordenan los textos por largo antes de armar los lotes para no gastar cómputo en padding.

In [ ]:
from transformers import AutoTokenizer, AutoModel

BERT_ID = 'distilbert-base-uncased'
tok_bert = AutoTokenizer.from_pretrained(BERT_ID)
bert = AutoModel.from_pretrained(BERT_ID).to(device).eval()

@torch.no_grad()
def bert_texto(textos):
    enc = tok_bert(textos, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
    h = bert(**enc).last_hidden_state                       # (B, T, 768)
    m = enc['attention_mask'].unsqueeze(-1).float()
    return ((h * m).sum(1) / m.sum(1)).float().cpu().numpy()

ruta_bert = f'{EMB}/distilbert_txt.npy'

if os.path.exists(ruta_bert):
    bert_txt = np.load(ruta_bert)
    print("Ya existía, se carga:", bert_txt.shape)
else:
    textos = df['sinopsis'].tolist()
    orden = np.argsort([len(t) for t in textos])
    bert_txt = np.zeros((N, 768), dtype=np.float32)
    for s in range(0, N, 64):
        idx = orden[s:s + 64]
        bert_txt[idx] = bert_texto([textos[i] for i in idx])
    np.save(ruta_bert, bert_txt)
    print("Guardado:", ruta_bert, bert_txt.shape)

## 5. Chequeos de calidad

Antes de usar los embeddings hay que comprobar que tienen sentido:

1. **Integridad:** formas, sin NaN, sin filas en cero.
2. **Alineamiento CLIP:** si los vectores de imagen y texto están bien emparejados, cada póster
   debería encontrar *su* sinopsis entre muchas mejor que al azar (recuperación top-k).
3. **Primer vistazo a la brecha modal:** similitud coseno de pares reales vs pares al azar, y
   distancia entre los centroides de cada modalidad (Liang et al., 2022). Es solo una mirada
   preliminar, sin IC: el análisis formal va en otro notebook.

In [ ]:
for nombre, X in [('clip_img', clip_img), ('clip_txt', clip_txt), ('distilbert_txt', bert_txt)]:
    normas = np.linalg.norm(X, axis=1)
    print(f"{nombre:15s} forma={X.shape}  NaN={np.isnan(X).sum()}  "
          f"filas_cero={(normas == 0).sum()}  norma: media={normas.mean():.2f} min={normas.min():.2f}")

In [ ]:
def l2(X): return X / np.linalg.norm(X, axis=1, keepdims=True)

zi, zt = l2(clip_img), l2(clip_txt)
test = np.where(df['particion'] == 'test')[0]

# Recuperación póster -> sinopsis entre 1000 películas de test
rng = np.random.default_rng(SEED)
sub = rng.choice(test, size=1000, replace=False)
sim = zi[sub] @ zt[sub].T                                       # (1000, 1000)
rango = (sim > np.diag(sim)[:, None]).sum(axis=1)               # 0 = su sinopsis quedó primera
for k in [1, 5, 10]:
    print(f"top-{k:<2d}: {(rango < k).mean():6.1%}   (azar: {k / len(sub):.1%})")

# Brecha modal (preliminar)
pares_reales = (zi * zt).sum(1)
pares_azar = (zi * zt[rng.permutation(N)]).sum(1)
brecha = np.linalg.norm(zi.mean(0) - zt.mean(0))
print(f"\ncos(pares reales) = {pares_reales.mean():.3f} | cos(pares al azar) = {pares_azar.mean():.3f}")
print(f"Distancia entre centroides imagen-texto: {brecha:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(pares_azar, bins=80, alpha=0.6, label='pares al azar')
ax.hist(pares_reales, bins=80, alpha=0.6, label='pares reales')
ax.set_xlabel('similitud coseno póster-sinopsis (CLIP)'); ax.set_ylabel('películas')
ax.legend(); plt.tight_layout(); plt.show()

## 6. Registro de la extracción

Se guarda un `info.json` con todo lo necesario para reproducir los archivos: modelos, versiones
y decisiones de preprocesamiento.

In [ ]:
info = {
    'fecha': datetime.datetime.now().isoformat(timespec='seconds'),
    'n_filas': int(N),
    'orden_filas': 'fila_hdf5 de mmimdb_meta.parquet',
    'versiones': {'transformers': transformers.__version__, 'torch': torch.__version__},
    'clip_img': {'modelo': CLIP_ID, 'salida': 'visual_projection(pooler_output), sin normalizar',
                 'preproceso': 'VGG->RGB, relleno a 256x256 con media de CLIP, CLIPImageProcessor'},
    'clip_txt': {'modelo': CLIP_ID, 'salida': 'text_projection(pooler_output)',
                 'preproceso': f'fragmentos de {UTILES}+2 tokens, cada uno normalizado L2, promedio por película'},
    'distilbert_txt': {'modelo': BERT_ID, 'salida': 'mean pooling de last_hidden_state con attention_mask',
                       'preproceso': 'truncado a 512 tokens'},
}
with open(f'{EMB}/info.json', 'w') as fh:
    json.dump(info, fh, indent=2, ensure_ascii=False)
!ls -lh {EMB}